In [6]:
import os
import sys
os.environ["KERAS_BACKEND"] = "torch"
sys.path.append("./src")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from sklearn.model_selection import train_test_split
import keras
from keras.metrics import Recall, Precision
from sklearn.metrics import confusion_matrix
from keras.callbacks import ModelCheckpoint
from keras.models import load_model

from models import *
from utils import *


# -------------------
# CONFIG
# -------------------
DATA_ROOT = "processed_data"
Multi_scale_patch = True
IMAGE_FOLDER = os.path.join(DATA_ROOT, "images")
LABELS_FILE = os.path.join(DATA_ROOT, "labels.csv")

MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)


CONFIG = {
    "img_size": (255, 255, 1),
    "batch_size": 32,
    "epochs": 30,
    "learning_rate": 1e-5,
    "model_name": "multiscale",   # 👈 change this to switch model
}


In [7]:
list_models()

Available models:
- baseline
- deeper
- deeperv2
- deeperv3
- FCN
- multiscale
- resnet50
- resnet18


In [ ]:
# -------------------
# HELPERS
# -------------------
def compile_model(model, config):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=config["learning_rate"]),
        loss='binary_crossentropy',
        metrics=['accuracy', Recall(name="recall"), Precision(name="precision")]
    )
    return model

def get_model_path(config):
    model_name = config["model_name"]
    
    
    ps = config["img_size"][0]


    filename = f"{model_name}_ps{ps}.keras"
    return os.path.join(MODEL_DIR, filename)

In [9]:
# -------------------
# LOAD LABELS
# -------------------
df = pd.read_csv(LABELS_FILE)

print("Total samples:", len(df))
print(df['label'].value_counts())

# -------------------
# SPLIT
# -------------------
train_df, temp_df = train_test_split(
    df, test_size=0.3, stratify=df['label'], random_state=42
)

val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42
)

# -------------------
# LOAD INTO MEMORY
# -------------------
X_train, Y_train = load_data(train_df, DATA_ROOT, Multi_scale_patch)
X_val, Y_val = load_data(val_df, DATA_ROOT, Multi_scale_patch)
X_test, Y_test = load_data(test_df, DATA_ROOT, Multi_scale_patch)


if Multi_scale_patch:
    print("Train balance:", np.mean(Y_train))
    print("Train:", X_train[0].shape)
    print("Train:", X_train[1].shape)
    print("Val:", X_val[0].shape)
    print("Val:", X_val[1].shape)
    print("Test:", X_test[0].shape)
    print("Test:", X_test[1].shape)
else:
    print("Train:", X_train.shape)
    print("Val:", X_val.shape)
    print("Test:", X_test.shape)


Total samples: 2632
label
1    1316
0    1316
Name: count, dtype: int64
Train balance: 0.5
Train: (1842, 255, 255, 1)
Train: (1842, 255, 255, 1)
Val: (395, 255, 255, 1)
Val: (395, 255, 255, 1)
Test: (395, 255, 255, 1)
Test: (395, 255, 255, 1)


In [10]:
# -------------------
# Model training
# -------------------
# build model
model = get_model(CONFIG["model_name"],CONFIG["img_size"]) 

# compile
model = compile_model(model, CONFIG)

model.summary()

model_path = get_model_path(CONFIG)

checkpoint = ModelCheckpoint(
    filepath=model_path,
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

# train
if Multi_scale_patch:
    history = model.fit(
        [X_train[0],X_train[1]], Y_train,
        validation_data=([X_val[0],X_val[1]], Y_val),
        epochs=CONFIG["epochs"],
        batch_size=CONFIG["batch_size"],
        callbacks=[checkpoint]
    )
else:
    history = model.fit(
    X_train, Y_train,
        validation_data=(X_val, Y_val),
        epochs=CONFIG["epochs"],
        batch_size=CONFIG["batch_size"],
        callbacks=[checkpoint]
    )

print("Loading best model from:", model_path)
model = load_model(model_path)

# -------------------
# PREDICTIONS
# -------------------
if Multi_scale_patch:
    y_pred_probs = model.predict([X_test[0], X_test[1]])
else:
    y_pred_probs = model.predict(X_test)

# -------------------
# THRESHOLD
# -------------------
threshold = 0.5
y_pred = (y_pred_probs > threshold).astype(int).flatten()
y_true = Y_test.astype(int)

# -------------------
# PER-LABEL ACCURACY
# -------------------
for label in [0, 1]:
    idx = (y_true == label)
    
    if np.sum(idx) == 0:
        print(f"Label {label}: no samples")
        continue

    acc = np.mean(y_pred[idx] == y_true[idx])
    print(f"Accuracy for label {label}: {acc:.4f} (n={np.sum(idx)})")

cm = confusion_matrix(y_true, y_pred)


# pretty print
print("Confusion Matrix:")
print(cm)

tn, fp, fn, tp = cm.ravel()

print(f"\nTrue Negatives: {tn}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"True Positives: {tp}")
# test
model.evaluate(X_test, Y_test)

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 255, 255,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_3       │ (None, 255, 255,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 255, 255,  │        320 │ input_layer_2[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_9 (Conv2D)   │ (None, 255, 255,  │        320 │ input_layer_3[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 255, 255,  │        128 │ conv2d_6[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 255, 255,  │        128 │ conv2d_9[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_6        │ (None, 255, 255,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_9        │ (None, 255, 255,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_4     │ (None, 127, 127,  │          0 │ activation_6[0][… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_6     │ (None, 127, 127,  │          0 │ activation_9[0][… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 127, 127,  │     18,496 │ max_pooling2d_4[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_10 (Conv2D)  │ (None, 127, 127,  │     18,496 │ max_pooling2d_6[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 127, 127,  │        256 │ conv2d_7[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 127, 127,  │        256 │ conv2d_10[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_7        │ (None, 127, 127,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_10       │ (None, 127, 127,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_5     │ (None, 63, 63,    │          0 │ activation_7[0][

 Total params: 203,649 (795.50 KB)

 Trainable params: 202,753 (792.00 KB)

 Non-trainable params: 896 (3.50 KB)

Epoch 1/30
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step - accuracy: 0.6451 - loss: 0.6250 - precision: 0.6289 - recall: 0.7301
Epoch 1: val_loss improved from None to 0.69011, saving model to models\multiscale_ps255.keras

Epoch 1: finished saving model to models\multiscale_ps255.keras
58/58 ━━━━━━━━━━━━━━━━━━━━ 8s 128ms/step - accuracy: 0.6819 - loss: 0.5920 - precision: 0.6667 - recall: 0.7275 - val_accuracy: 0.5013 - val_loss: 0.6901 - val_precision: 0.5013 - val_recall: 1.0000
Epoch 2/30
58/58 ━━━━━━━━━━━━━━━━━━━━ 0s 157ms/step - accuracy: 0.7489 - loss: 0.5168 - precision: 0.7555 - recall: 0.7400
Epoch 2: val_loss improved from 0.69011 to 0.67793, saving model to models\multiscale_ps255.keras

Epoch 2: finished saving model to models\multiscale_ps255.keras
58/58 ━━━━━━━━━━━━━━━━━━━━ 10s 165ms/step - accuracy: 0.7492 - loss: 0.5145 - precision: 0.7641 - recall: 0.7210 - val_accuracy: 0.5013 - val_loss: 0.6779 - val_precision: 0.5013 - val_recall: 1.0000
Epoch 3/30
58/58 ━━━━━━━━━━━━━━━

[0.49383285641670227, 0.7569620013237, 0.7563451528549194, 0.7563451528549194]

In [11]:
# -------------------
# CONFIG (adjust these)
# -------------------
Load_from_ZIP = True
zip_path = "data/vindr-mammo-a-large-scale-benchmark-dataset-for-computer-aided-detection-and-diagnosis-in-full-field-digital-mammography-1.0.0.zip"
image_folder = "data/train_images/"
patch_size = (255, 255)
stride = 64

# -------------------
# LOAD CSV (must exist already)
# -------------------
df = pd.read_csv("data/finding_annotations.csv")
df = df[['image_id','finding_categories','xmin','ymin','xmax','ymax']]

# -------------------
# BUILD IMAGE INDEX
# -------------------
image_index, z = build_image_index(
    image_folder=image_folder,
    zip_path=zip_path,
    load_from_zip=Load_from_ZIP
)

# -------------------
# LOAD BEST MODEL
# -------------------
model_path = get_model_path(CONFIG)
print("Loading best model from:", model_path)
model = load_model(model_path)

# -------------------
# HEATMAP GENERATION
# -------------------
def generate_heatmap(model, img):
    h, w = img.shape
    heatmap = np.zeros((h, w))
    counts = np.zeros((h, w))

    for y in range(0, h - patch_size[0] + 1, stride):
        for x in range(0, w - patch_size[1] + 1, stride):

            if Multi_scale_patch:
                p_small, p_large = extract_multiscale_patch(img, x, y, patch_size[0])

                p_small = np.expand_dims(p_small, axis=(0, -1))
                p_large = np.expand_dims(p_large, axis=(0, -1))

                pred = model.predict([p_small, p_large], verbose=0)[0][0]

            else:
                patch = img[y:y+patch_size[0], x:x+patch_size[1]]
                patch = np.expand_dims(patch, axis=(0, -1))

                pred = model.predict(patch, verbose=0)[0][0]

            # optional threshold stabilization
            pred = 0 if pred < 0.6 else pred

            heatmap[y:y+patch_size[0], x:x+patch_size[1]] += pred
            counts[y:y+patch_size[0], x:x+patch_size[1]] += 1

    heatmap = heatmap / (counts + 1e-6)

    return heatmap

# -------------------
# VISUALIZATION
# -------------------
def show_heatmap(model, image_id):
    img = load_image(
    image_id=image_id,
    image_index=image_index,
    load_from_zip=Load_from_ZIP,
    zip_file=z
)

    if img is None:
        print("Image not found")
        return

    heatmap = generate_heatmap(model, img)

    plt.figure(figsize=(10,10))
    plt.imshow(img, cmap='gray')

    # overlay heatmap
    plt.imshow(heatmap, cmap='jet', alpha=0.4)

    # draw GT boxes
    rows = df[df['image_id'] == image_id]

    for _, row in rows.iterrows():
        if row['finding_categories'] == "['Mass']" and not np.isnan(row['xmin']):
            x = row['xmin']
            y = row['ymin']
            w = row['xmax'] - row['xmin']
            h = row['ymax'] - row['ymin']

            rect = patches.Rectangle(
                (x, y), w, h,
                linewidth=2,
                edgecolor='lime',
                facecolor='none'
            )
            plt.gca().add_patch(rect)

    plt.title(f"Heatmap for {image_id}")
    plt.axis('off')
    plt.show()

def get_random_mass_image_id(df):
    # keep only rows with Mass and valid boxes
    mass_df = df[
        (df['finding_categories'] == "['Mass']") &
        (~df['xmin'].isna())
    ]

    if len(mass_df) == 0:
        print("No Mass images found in dataframe")
        return None

    # get unique image_ids
    image_ids = mass_df['image_id'].unique()

    # pick one randomly
    return np.random.choice(image_ids)

def get_random_no_mass_image_id(df):
    grouped = df.groupby('image_id')

    no_mass_ids = []

    for image_id, group in grouped:
        # no valid bounding boxes = no findings
        if group['xmin'].isna().all():
            no_mass_ids.append(image_id)

    if len(no_mass_ids) == 0:
        print("No pure 'No Finding' images found")
        return None

    return np.random.choice(no_mass_ids)
# -------------------
# RUN
# -------------------

mass_id = get_random_mass_image_id(df)
no_mass_id = get_random_no_mass_image_id(df)

if mass_id is not None:
    print("\n--- MASS IMAGE ---")
    print("Selected image:", mass_id)
    show_heatmap(model, mass_id)

if no_mass_id is not None:
    print("\n--- NO MASS IMAGE ---")
    print("Selected image:", no_mass_id)
    show_heatmap(model, no_mass_id)

Loading best model from: models\multiscale_ps128.keras

--- MASS IMAGE ---
Selected image: eaf9352abcc67846d1f69f4d21474e2f


ValueError: Input 0 with name 'input_layer_2' of layer 'functional_1' is incompatible with the layer: expected shape=(None, 128, 128, 1), found shape=(1, 255, 255)